# I09 - Advanced Transformer Architectures

**Level:** Intermediate

---

## Learning Objectives

By the end of this lesson, you will:
1. Understand BERT architecture and pre-training
2. Learn GPT variants (GPT-2, GPT-3 concepts)
3. Explore T5 and BART models
4. Deep dive into positional encoding
5. Compare different transformer architectures

---

## Prerequisites

- Completed B11 (Attention and Transformers)
- Understanding of self-attention
- Familiarity with transformer basics

---

In [ ]:
# Import required libraries
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print(f"TensorFlow version: {tf.__version__}")

## 1. BERT: Bidirectional Encoder Representations from Transformers

### Key Innovation

**Previous models:** Unidirectional (left-to-right)
- GPT: Only sees previous tokens
- ELMo: Shallow bidirectional

**BERT:** Deep bidirectional
- Sees both left and right context
- Pre-trained on massive text corpus
- Fine-tuned for downstream tasks

### Architecture

**BERT-Base:**
- 12 transformer encoder layers
- 768 hidden dimensions
- 12 attention heads
- 110M parameters

**BERT-Large:**
- 24 transformer encoder layers
- 1024 hidden dimensions
- 16 attention heads
- 340M parameters

### Pre-training Tasks

**1. Masked Language Modeling (MLM):**
- Randomly mask 15% of tokens
- Predict masked tokens
- Example: "The [MASK] is blue" → "sky"

**2. Next Sentence Prediction (NSP):**
- Given two sentences A and B
- Predict if B follows A
- Binary classification task

In [ ]:
class BERTEncoder(layers.Layer):
    """Simplified BERT encoder layer"""
    
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation='gelu'),
            layers.Dense(embed_dim)
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)
    
    def call(self, inputs, training=False):
        # Multi-head attention
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        
        # Feed-forward network
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

def create_bert_style_model(vocab_size=30000, max_len=512, embed_dim=768, 
                           num_heads=12, ff_dim=3072, num_layers=12):
    """Create BERT-style model
    
    Args:
        vocab_size: Vocabulary size
        max_len: Maximum sequence length
        embed_dim: Embedding dimension
        num_heads: Number of attention heads
        ff_dim: Feed-forward dimension
        num_layers: Number of encoder layers
    """
    inputs = keras.Input(shape=(max_len,))
    
    # Token + Position embeddings
    token_emb = layers.Embedding(vocab_size, embed_dim)(inputs)
    pos_emb = layers.Embedding(max_len, embed_dim)(tf.range(start=0, limit=max_len, delta=1))
    x = token_emb + pos_emb
    
    # Stack of encoder layers
    for _ in range(num_layers):
        x = BERTEncoder(embed_dim, num_heads, ff_dim)(x)
    
    # Pooler (for classification tasks)
    pooled = layers.Lambda(lambda x: x[:, 0, :])(x)  # [CLS] token
    pooled = layers.Dense(embed_dim, activation='tanh')(pooled)
    
    model = keras.Model(inputs, [x, pooled], name='BERT_Style')
    return model

print("BERT-style model architecture defined")
print("\nKey features:")
print("- Bidirectional attention (sees full context)")
print("- Pre-trained with MLM + NSP")
print("- Fine-tuned for specific tasks")

---

## Key Takeaways

**Relevant UoA Courses:** COMPSCI 703, COMPSYS 721

1. BERT: bidirectional encoder, masked language modeling (MLM)
2. GPT: unidirectional decoder, causal language modeling
3. T5: encoder-decoder, text-to-text framework
4. BART: denoising autoencoder, combines BERT and GPT ideas
5. Positional encoding: sinusoidal or learned embeddings

---

## Exam Preparation Guide

### Essential Concepts for Exams

- Compare architectures: BERT (encoder), GPT (decoder), T5 (encoder-decoder)
- Understand training objectives: MLM (BERT), CLM (GPT), span corruption (T5)
- Know use cases: BERT (classification), GPT (generation), T5 (seq2seq)
- Explain masked attention vs causal attention
- Common question: Design transformer for specific task (classification vs generation)

### Common Mistakes to Avoid

- ❌ Confusing BERT and GPT architectures
- ❌ Not understanding bidirectional vs unidirectional attention
- ❌ Forgetting that BERT can't generate text autoregressively

### Practice Problems

1. Compare BERT vs GPT: architecture, training, use cases
2. Why can't BERT generate text like GPT?
3. Design transformer for machine translation: encoder-decoder or decoder-only?

### How This Helps Your UoA Courses

**COMPSCI 703, COMPSYS 721:**
- Provides hands-on implementation of theoretical concepts
- Practice problems similar to exam questions
- Reinforces lecture material with code examples
- Helps build intuition for complex topics

### Study Tips for Advanced Topics

1. **Connect to Fundamentals**: Link advanced concepts to basics
2. **Read Papers**: Understand state-of-the-art approaches
3. **Implement from Scratch**: Deepens understanding
4. **Compare Approaches**: Know trade-offs between methods
5. **Real-World Applications**: Understand practical use cases

### Exam Question Types

- **Conceptual**: Explain advanced mechanisms and why they work
- **Comparison**: Compare multiple approaches, trade-offs
- **Design**: Design system for specific requirements
- **Analysis**: Analyze experimental results, identify issues
- **Application**: Apply techniques to novel problems

---


---

## Learning Progress Tracker

Use this section to track your learning progress for this lesson.

### Completion Status
- [ ] Lesson completed
- [ ] All code cells executed successfully
- [ ] Understood all key concepts
- [ ] Completed practice exercises (if any)

### Dates
- **First Completed:** ____/____/____
- **Last Reviewed:** ____/____/____
- **Next Review:** ____/____/____ (Recommended: 1 week, 1 month, 3 months)

### Understanding Level
Rate your understanding (1-5): _____ / 5

- 1 = Need to review completely
- 2 = Understood basics, need more practice
- 3 = Good understanding, minor gaps
- 4 = Strong understanding, can explain to others
- 5 = Mastered, can apply in projects

### Notes & Reflections
```
Write your notes here:
- What concepts were challenging?
- What was interesting or surprising?
- How can you apply this in projects?
- Questions to explore further?




```

### Key Concepts to Remember (I09)
- [ ] BERT architecture
- [ ] GPT variants
- [ ] T5 model
- [ ] BART

---